# CMIP6: quickstart

Pull one model's projection of near-surface air temperature under SSP5-8.5
over the UK for 2050, via the Copernicus Climate Data Store.

See [`docs/cmip6/README.md`](../docs/cmip6/README.md) for the full
reference. Prerequisites: CDS account, CMIP6 licence accepted, `~/.cdsapirc`
credentials.

For research-grade ensemble analysis consider `intake-esm` plus the Pangeo
Zarr mirror; the CDS is optimised for quickstart-scale single-model pulls.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
MODEL = "mpi_esm1_2_lr"
EXPERIMENT = "ssp5_8_5"  # or ssp2_4_5, ssp1_2_6, historical, etc.
VARIABLE = "near_surface_air_temperature"
TEMPORAL_RESOLUTION = "monthly"
YEARS = ["2050"]
MONTHS = [f"{m:02d}" for m in range(1, 13)]
BBOX = [61, -8, 49, 2]  # UK
OUTPUT_DIR = "../data/cmip6"
OUTPUT_FILENAME = "cmip6_quickstart.nc"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request

CMIP6 requests use different keys from ERA5: `temporal_resolution`,
`experiment`, `model`, and no `product_type`. Output is zipped NetCDF
(one or more files per request). Expect queue times to be less predictable
than ERA5 because the CDS dispatches CMIP6 jobs to ESGF nodes.

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
zip_path = output_dir / (OUTPUT_FILENAME + ".zip")
output_path = output_dir / OUTPUT_FILENAME

request = {
    "temporal_resolution": TEMPORAL_RESOLUTION,
    "experiment": EXPERIMENT,
    "variable": VARIABLE,
    "model": MODEL,
    "year": YEARS,
    "month": MONTHS,
    "area": BBOX,
    "data_format": "netcdf",
    "download_format": "zip",
}

client = cdsapi.Client()
client.retrieve("projections-cmip6", request).download(str(zip_path))

# Unpack the zip to the output directory
extract_dir = output_dir / "_extracted"
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

nc_files = list(extract_dir.glob("*.nc"))
print(f"Unpacked {len(nc_files)} NetCDF file(s): {[f.name for f in nc_files]}")

## Open and inspect

CMIP6 files may come split by time chunk; merge with `xr.open_mfdataset`.
The variable appears under its CMIP6 short name (`tas`) inside the file,
not the CDS request string.

In [ ]:
ds = xr.open_mfdataset(nc_files, combine="by_coords")
print(ds)
print("\nData variables:", list(ds.data_vars))
print("Calendar:", ds["time"].attrs.get("calendar", "standard"))

## Plot the projected annual cycle

With 12 monthly values for 2050 over a small UK bbox, an area-mean annual
cycle is the natural view. This is one model's projection under one
scenario - interpret as a point estimate within an ensemble spread, not a
forecast.

In [ ]:
tas = ds["tas"].mean(dim=["lat", "lon"]) - 273.15

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tas["time"].values, tas.values, marker="o")
ax.set_xlabel("Month")
ax.set_ylabel("UK area-mean 2 m temperature (C)")
ax.set_title(f"CMIP6 {MODEL}, {EXPERIMENT}, {YEARS[0]} annual cycle")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/cmip6/README.md`](../docs/cmip6/README.md)
- **Multi-model ensemble**: change `MODEL` to a list. Be mindful of request
  size; asking for all ~60 models at once will fail.
- **Multi-scenario comparison**: loop over SSPs (`ssp1_2_6`, `ssp2_4_5`,
  `ssp3_7_0`, `ssp5_8_5`) with historical as the baseline.
- **Regridding across models**: use `xESMF` or `CDO` to bring models onto
  a common grid before computing an ensemble mean.
- **Calendar handling**: for multi-model time-series work, convert to a
  common calendar with `xr.convert_calendar` or use `cftime` throughout.
- **Research-grade ensemble analysis**: `intake-esm` plus the Pangeo
  CMIP6 Zarr mirror on Google Cloud is far more efficient for hundreds
  of files.